# 19 · Biomarker and genetic-subgroup merge

Pulls baseline DaTscan, CSF biomarkers, APOE-ε4, α-syn SAA, and the PPMI `subgroup` variable (Sporadic / LRRK2 / GBA / SNCA / PRKN / LRRK2+GBA / PINK1) and merges them onto the patient-anchor feature frame.

In [1]:
source("helpers/pain_helpers.R")
suppressPackageStartupMessages({ library(dplyr); library(readr); library(tidyr); library(broom); library(purrr) })
curated <- readr::read_csv(file.path(PROJECT_ROOT, "PPMI_Curated_Data_Cut_Public_20241211.csv"),
                           show_col_types = FALSE)
cat("Curated rows:", nrow(curated), "  unique patients:", dplyr::n_distinct(curated$PATNO), "\n")

Warning message:
“package ‘broom’ was built under R version 4.5.2”


Curated rows: 14473   unique patients: 3866 


In [2]:
bio_cols <- c("mean_caudate","mean_putamen","mean_striatum","con_putamen","ips_putamen",
              "abeta","tau","ptau","asyn","NFL_CSF","nfl_serum","APOE_e4","CSFSAA","subgroup")
bio <- curated %>% dplyr::filter(COHORT == 1) %>%
  dplyr::arrange(PATNO, visit_date) %>%
  dplyr::group_by(PATNO) %>%
  dplyr::summarise(dplyr::across(dplyr::all_of(bio_cols), ~ .x[!is.na(.x)][1]), .groups = "drop")
cat("Baseline biomarker rows:", nrow(bio), "\n")
cat("Subgroup counts:\n"); print(dplyr::count(bio, subgroup, sort = TRUE))
cat("APOE_e4 counts:\n"); print(table(bio$APOE_e4, useNA = "ifany"))
cat("CSFSAA counts:\n"); print(table(bio$CSFSAA, useNA = "ifany"))

Baseline biomarker rows: 1441 


Subgroup counts:


# A tibble: 7 × 2
  subgroup        n
  <chr>       <int>
1 Sporadic PD  1081
2 LRRK2         178
3 GBA           110
4 SNCA           37
5 PRKN           25
6 LRRK2 + GBA     9
7 PINK1           1


APOE_e4 counts:



   0    1    2 <NA> 
 630  188   16  607 


CSFSAA counts:



   0    1    2    3 <NA> 
 150 1092    7    8  184 


In [3]:
dat <- readRDS(file.path(OUT_OBJ, "patient_anchor_features.rds"))
enriched <- dat %>% dplyr::left_join(bio, by = "PATNO")
cat("Enriched n:", nrow(enriched), "\n")
cov <- enriched %>% dplyr::mutate(arm = dplyr::if_else(will_receive_dbs, "DBS", "Never-DBS")) %>%
  dplyr::group_by(arm) %>%
  dplyr::summarise(
    n = dplyr::n(),
    n_putamen = sum(!is.na(mean_putamen)),
    n_asyn    = sum(!is.na(asyn)),
    n_nfl     = sum(!is.na(NFL_CSF)),
    n_saa     = sum(!is.na(CSFSAA)),
    n_apoe    = sum(!is.na(APOE_e4)),
    n_subgrp  = sum(!is.na(subgroup)),
    .groups = "drop"
  )
print(cov)
save_object(enriched, "patient_anchor_features_enriched")
save_table(cov, "biomarker_coverage_by_arm")

Enriched n: 642 


# A tibble: 2 × 8
  arm           n n_putamen n_asyn n_nfl n_saa n_apoe n_subgrp
  <chr>     <int>     <int>  <int> <int> <int>  <int>    <int>
1 DBS          67        67     53    30    62     67       67
2 Never-DBS   575       542    330   176   512    467      557


In [4]:
bio_vars <- c("mean_putamen","mean_caudate","con_putamen","asyn","NFL_CSF","nfl_serum",
              "abeta","tau","ptau","APOE_e4")
uni_bio <- purrr::map_dfr(bio_vars, function(v) {
  d <- enriched %>% dplyr::select(worsened, !!v) %>% tidyr::drop_na()
  if (nrow(d) < 30) return(tibble::tibble(variable = v, n = nrow(d), or = NA, lci = NA, uci = NA, p = NA))
  fit <- tryCatch(stats::glm(stats::as.formula(paste("worsened ~", v)), data = d, family = "binomial"), error = function(e) NULL)
  if (is.null(fit)) return(tibble::tibble(variable = v, n = nrow(d), or = NA, lci = NA, uci = NA, p = NA))
  co <- suppressMessages(broom::tidy(fit, conf.int = TRUE)) %>% dplyr::filter(term == v)
  tibble::tibble(variable = v, n = nrow(d), or = exp(co$estimate),
                 lci = exp(co$conf.low), uci = exp(co$conf.high), p = co$p.value)
})
print(uni_bio)
save_table(uni_bio, "biomarker_univariate_or")

# A tibble: 10 × 6
   variable         n    or   lci   uci      p
   <chr>        <int> <dbl> <dbl> <dbl>  <dbl>
 1 mean_putamen   609 0.908 0.445 1.76  0.782 
 2 mean_caudate   609 0.852 0.583 1.24  0.405 
 3 con_putamen    607 0.876 0.396 1.79  0.729 
 4 asyn           383 0.999 0.999 1.000 0.0623
 5 NFL_CSF        206 1.00  0.999 1.01  0.0701
 6 nfl_serum      438 1.01  0.984 1.04  0.384 
 7 abeta          432 0.999 0.998 1.000 0.0161
 8 tau            441 1.00  0.996 1.00  0.930 
 9 ptau           429 0.997 0.948 1.04  0.896 
10 APOE_e4        534 0.989 0.581 1.61  0.964 


In [5]:
sg <- enriched %>% dplyr::filter(!is.na(subgroup)) %>%
  dplyr::mutate(sg = dplyr::case_when(
    grepl("LRRK2", subgroup) ~ "LRRK2",
    grepl("GBA", subgroup)   ~ "GBA",
    subgroup == "SNCA"       ~ "SNCA",
    subgroup == "Sporadic PD"~ "Sporadic",
    TRUE                     ~ "Other"
  ),
  arm = dplyr::if_else(will_receive_dbs, "DBS", "Never-DBS"))
tab <- sg %>% dplyr::group_by(sg, arm) %>%
  dplyr::summarise(n = dplyr::n(), mean_delta = mean(delta, na.rm = TRUE),
                   pct_worsen = mean(worsened == 1), .groups = "drop")
print(tab, n = 20)
save_table(tab, "pain_by_subgroup_arm")

# A tibble: 8 × 5
  sg       arm           n mean_delta pct_worsen
  <chr>    <chr>     <int>      <dbl>      <dbl>
1 GBA      DBS          10     0.0787      0.1  
2 GBA      Never-DBS    48    -0.0481      0.104
3 LRRK2    DBS          22     0.248       0.227
4 LRRK2    Never-DBS    91     0.0653      0.198
5 Other    DBS           1     0.5         0    
6 Other    Never-DBS    13     0.0574      0.154
7 Sporadic DBS          34    -0.0710      0.147
8 Sporadic Never-DBS   405     0.0917      0.148
